Data sampling with a sliding window with number data

In [ ]:
from importlib.metadata import version  # 用于查询已安装第三方库的版本号
import torch  # PyTorch 深度学习框架，本notebook用它做张量（tensor）运算

print("torch version:", version("torch"))  # 打印当前环境安装的 torch 版本，便于复现实验环境

In [ ]:
# 生成一份简单的数字序列文件，用连续整数代替真实文本的 token，
# 这样无需依赖分词器（tokenizer）即可直观演示滑动窗口如何切分数据
with open("number-data.txt", "w", encoding="utf-8") as f:
    for number in range(1001):  # 写入 0 到 1000，共 1001 个数字
        f.write(f"{number} ")  # 每个数字后面加一个空格作为分隔符

In [ ]:
from torch.utils.data import Dataset, DataLoader  # PyTorch 提供的数据集基类和数据加载器
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []   # 保存所有输入序列，每个元素是形状为 (max_length,) 的一维张量
        self.target_ids = []  # 保存所有目标序列，与输入序列一一对应，整体在位置上错开一位
         # Modification
        # token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        # 注：这里没有真正调用 tokenizer 编码文本，而是直接把空格分隔的数字字符串转成整数列表，
        # 把每个数字当作一个“token id”，方便脱离真实分词器、直观展示滑动窗口的切分逻辑
        token_ids=[int(i) for i in txt.strip().split()]
        # Use a sliding window to chunk the book into overlapping sequences of max_length
        # 用滑动窗口把整段 token 序列切分成多个长度为 max_length 的（可能相互重叠的）子序列；
        # range 的终止边界预留了 max_length，保证切片时不会越过序列末尾
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk = token_ids[i:i + max_length]           # 输入块：从位置 i 开始，取连续 max_length 个 token
            target_chunk = token_ids[i + 1: i + max_length + 1] # 目标块：相对输入块整体向右错位一位，即“预测下一个 token”
            self.input_ids.append(torch.tensor(input_chunk))    # 转成形状为 (max_length,) 的一维张量
            self.target_ids.append(torch.tensor(target_chunk))  # 同样是形状为 (max_length,) 的一维张量
    def __len__(self):
        return len(self.input_ids)  # 数据集大小 = 滑动窗口一共切出的样本（输入/目标对）数量
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]  # 按索引返回一对 (输入张量, 目标张量)

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
     # Initialize the tokenizer
    # tokenizer = tiktoken.get_encoding("gpt2")
    # 本示例操作的是数字序列，不需要真实分词器，故直接置为 None（GPTDatasetV1 内部也未使用它）
    tokenizer = None
     # Create dataset
    # 用滑动窗口数据集，把原始数字序列切分成若干 (输入, 目标) 样本对
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    # DataLoader 负责按 batch_size 把样本打包成批次，可控制是否打乱顺序（shuffle）、
    # 是否丢弃最后不满一个 batch 的数据（drop_last），以及是否使用多进程加载（num_workers）
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
# 重新读取刚才写入的数字文件，作为后续 DataLoader 的原始文本输入
with open("number-data.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
# batch_size=1：每个 batch 只含 1 个样本；max_length=4：每个序列长度为 4；
# stride=1：滑动窗口每次只移动 1 步，因此相邻样本高度重叠；shuffle=False：保持原始顺序便于观察
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

data_iter = iter(dataloader)   # 创建迭代器，用于逐个取出 batch
first_batch = next(data_iter)  # 取出第一个 batch：一个 (输入张量, 目标张量) 的元组，形状均为 (1, 4)
print(first_batch)
# 例如输出 [tensor([[0, 1, 2, 3]]), tensor([[1, 2, 3, 4]])]：
# target 张量相对 input 张量整体向右错位一位，体现“用当前 token 预测下一个 token”

In [ ]:
# stride=1，所以第二个 batch 相对第一个 batch 只向右滑动了 1 步
# 例如 input 由 [0,1,2,3] 变为 [1,2,3,4]，直观体现滑动窗口样本间的重叠
second_batch = next(data_iter)
print(second_batch)

In [ ]:
# 继续取下一个 batch，滑动窗口继续向右移动 1 步
third_batch = next(data_iter)
print(third_batch)

In [ ]:
# 遍历整个 dataloader；pass 表示循环体内不做任何处理，仅用于把迭代推进到最后一个 batch
for batch in dataloader:
    pass

last_batch = batch  # for 循环结束后，batch 变量仍保留最后一次迭代得到的 batch（即最后一个样本）
print(last_batch)

In [ ]:
# batch_size=2：每个 batch 打包 2 个样本；stride=4 恰好等于 max_length，
# 意味着相邻窗口不再重叠，每个数字只会出现在唯一一个输入序列中
dataloader = create_dataloader_v1(raw_text, batch_size=2, max_length=4, stride=4, shuffle=False)

for inputs, targets in dataloader:
    pass  # 只遍历到最后一个 batch，不做其他处理

# 打印最后一个 batch 的输入/目标张量，形状均为 (batch_size, max_length) = (2, 4)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

In [ ]:
torch.manual_seed(123)  # 固定随机种子，使下面 shuffle=True 打乱样本的结果可复现
# shuffle=True：这次会在每个 epoch 开始时随机打乱样本顺序，模拟真实训练场景
dataloader = create_dataloader_v1(raw_text, batch_size=2, max_length=4, stride=4, shuffle=True)

for inputs, targets in dataloader:
    pass  # 遍历到最后一个 batch

# 打乱后取到的最后一个 batch，输入/目标张量形状依然是 (batch_size, max_length) = (2, 4)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)